# plot_future_hydrograph notebook

This notebook embeds the full code from `plot_future_hydrograph.py` and runs `main()` directly.

In [ ]:
from pathlib import Path
import os

# Optional setup cell. Main code cell also sets project root.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)
print(f"Working directory: {ROOT}")

In [ ]:
from pathlib import Path
import os

# Notebook: ensure project root (run big cell alone).
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)

"""
Plot future simulated discharge with rainplusmelt on an inverted secondary y-axis
(same single-panel layout as the 2006–2009 zoom hydrographs). No observed discharge.

Overlays seasonal Q20, Q50, and Q80 thresholds (from compute_seasonal_thresholds.py) mapped by
calendar day-of-year, so flow relative to the baseline climatology is visible.

Run only when needed (PNG can be slow for ~95 years of daily bars):
  python plot_future_hydrograph.py --basin_id SE000197
  python plot_future_hydrograph.py --basin_id CH000127 --threshold_years 30

Interactive HTML (pan/zoom in browser; optional date window):
  python plot_future_hydrograph.py --basin_id SE000197 --interactive
  python plot_future_hydrograph.py --basin_id CH000127 --interactive --plot_start_date 2040-01-01 --plot_end_date 2050-12-31
"""

from __future__ import annotations

import argparse
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from calibration_io import basin_hydrograph_title
from hydrograph_interactive import build_overlay_hydrograph, write_interactive_html


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawTextHelpFormatter)
    p.add_argument("--root", type=Path, default=Path("."), help="Project root.")
    p.add_argument("--basin_id", required=True, help="Basin ID (column name in CSVs).")
    p.add_argument(
        "--q_csv",
        type=Path,
        default=None,
        help="Wide future Q CSV (default: <root>/Results/discharge_components_future/Q_total_all_basins.csv).",
    )
    p.add_argument(
        "--rain_csv",
        type=Path,
        default=None,
        help="Wide future rainplusmelt CSV (default: <root>/Results/forcing_future/rainplusmelt.csv).",
    )
    p.add_argument(
        "--q80_csv",
        type=Path,
        default=None,
        help="Q80 climatology wide table (default: <root>/Results/thresholds/Q80_daily_thresholds_last{years}y.csv).",
    )
    p.add_argument(
        "--q50_csv",
        type=Path,
        default=None,
        help="Q50 climatology wide table (default: same folder as Q80, Q50_daily_thresholds_last{years}y.csv).",
    )
    p.add_argument(
        "--q20_csv",
        type=Path,
        default=None,
        help="Q20 climatology wide table (default: same folder as Q80, Q20_daily_thresholds_last{years}y.csv).",
    )
    p.add_argument("--threshold_years", type=int, default=30, help="Suffix years for default Q80 filename.")
    p.add_argument(
        "--out_png",
        type=Path,
        default=None,
        help="Output PNG path (default: <root>/Results/hydrographs_future/hydrograph_future_{basin}.png).",
    )
    p.add_argument("--fig_width", type=float, default=12.0)
    p.add_argument("--fig_height", type=float, default=5.8)
    p.add_argument("--dpi", type=int, default=150)
    p.add_argument(
        "--interactive",
        action="store_true",
        help="Write zoomable HTML instead of (or in addition to) PNG.",
    )
    p.add_argument(
        "--also_png",
        action="store_true",
        help="With --interactive, also save the static PNG.",
    )
    p.add_argument(
        "--out_html",
        type=Path,
        default=None,
        help="Output HTML path (default: .../hydrograph_future_{basin}.html).",
    )
    p.add_argument("--plot_start_date", default="", help="Optional first day to plot (YYYY-MM-DD).")
    p.add_argument("--plot_end_date", default="", help="Optional last day to plot (YYYY-MM-DD).")
    return p.parse_known_args()[0]


def load_threshold_series(dates: pd.DatetimeIndex, wide: pd.DataFrame, basin_id: str) -> np.ndarray:
    """Map each calendar date to a threshold from ddmm-indexed wide table (Q80, Q50, Q20, …)."""
    if basin_id not in wide.columns:
        raise ValueError(f"Basin {basin_id!r} not in threshold table columns: {list(wide.columns)[:10]}...")
    lut = wide.set_index("ddmm")[basin_id]
    ddmm = pd.Series(dates.strftime("%d-%m"), dtype=str)
    leap = (dates.month == 2) & (dates.day == 29)
    ddmm = ddmm.mask(leap, "28-02")
    s = ddmm.map(lut)
    s = pd.to_numeric(s, errors="coerce")
    return s.ffill().bfill().to_numpy(dtype=float)


def main() -> None:
    args = parse_args()
    root = args.root.resolve()
    basin_id = args.basin_id

    q_csv = args.q_csv or (root / "Results" / "discharge_components_future" / "Q_total_all_basins.csv")
    rain_csv = args.rain_csv or (root / "Results" / "forcing_future" / "rainplusmelt.csv")
    q80_csv = args.q80_csv or (root / "Results" / "thresholds" / f"Q80_daily_thresholds_last{args.threshold_years}y.csv")
    thr_dir = q80_csv.parent
    q50_csv = args.q50_csv or (thr_dir / f"Q50_daily_thresholds_last{args.threshold_years}y.csv")
    q20_csv = args.q20_csv or (thr_dir / f"Q20_daily_thresholds_last{args.threshold_years}y.csv")

    for path, label in ((q_csv, "Q"), (rain_csv, "rainplusmelt"), (q80_csv, "Q80")):
        if not path.exists():
            raise FileNotFoundError(f"Missing {label} file: {path}")

    q_df = pd.read_csv(q_csv)
    rain_df = pd.read_csv(rain_csv)
    q80_wide = pd.read_csv(q80_csv)
    q50_wide = pd.read_csv(q50_csv) if q50_csv.exists() else None
    q20_wide = pd.read_csv(q20_csv) if q20_csv.exists() else None
    if q50_wide is None:
        print(f"Note: Q50 climatology not found ({q50_csv}); plot will omit Q50 overlay.")
    if q20_wide is None:
        print(f"Note: Q20 climatology not found ({q20_csv}); plot will omit Q20 overlay.")

    out_png = args.out_png or (root / "Results" / "hydrographs_future" / f"hydrograph_future_{basin_id}.png")

    q_df["date"] = pd.to_datetime(q_df["date"]).dt.normalize()
    rain_df["date"] = pd.to_datetime(rain_df["date"]).dt.normalize()

    if basin_id not in q_df.columns:
        raise ValueError(f"Basin {basin_id!r} not in {q_csv}")
    if basin_id not in rain_df.columns:
        raise ValueError(f"Basin {basin_id!r} not in {rain_csv}")

    q_snip = q_df[["date", basin_id]].rename(columns={basin_id: "q_sim"})
    p_snip = rain_df[["date", basin_id]].rename(columns={basin_id: "precip"})
    sub = q_snip.merge(p_snip, on="date", how="inner")
    q_all = pd.to_numeric(sub["q_sim"], errors="coerce")
    n_q_nan = int(q_all.isna().sum())
    if n_q_nan and n_q_nan / len(sub) > 0.01:
        first_ok = sub.loc[q_all.notna(), "date"]
        last_ok = first_ok.max() if not first_ok.empty else None
        print(
            f"Warning: {n_q_nan}/{len(sub)} days have NaN simulated Q "
            f"({100 * n_q_nan / len(sub):.1f}%). Plot will only show days with finite Q and P."
            + (f" Last finite Q date: {last_ok:%Y-%m-%d}." if last_ok is not None else "")
            + " Check future PET (tasmin > tasmax after bias correction can break Hargreaves)."
        )
    sub = sub.dropna(subset=["q_sim", "precip"])
    if sub.empty:
        raise ValueError("No overlapping dates between Q and rainplusmelt.")

    dates = pd.DatetimeIndex(sub["date"].values)
    sub = sub.copy()
    sub["q80"] = load_threshold_series(dates, q80_wide, basin_id)
    if q50_wide is not None:
        sub["q50"] = load_threshold_series(dates, q50_wide, basin_id)
    if q20_wide is not None:
        sub["q20"] = load_threshold_series(dates, q20_wide, basin_id)

    if args.plot_start_date.strip():
        sub = sub.loc[sub["date"] >= pd.to_datetime(args.plot_start_date)]
    if args.plot_end_date.strip():
        sub = sub.loc[sub["date"] <= pd.to_datetime(args.plot_end_date)]
    if sub.empty:
        raise ValueError("No data left after date filters.")

    title = basin_hydrograph_title(
        basin_id,
        date_start=f"{sub['date'].min():%Y-%m-%d}",
        date_end=f"{sub['date'].max():%Y-%m-%d}",
    )
    q80_legend = f"Q80 (baseline, last {args.threshold_years}y climatology)"
    q50_legend = f"Q50 (baseline, last {args.threshold_years}y climatology)"
    q20_legend = f"Q20 (baseline, last {args.threshold_years}y climatology)"

    if args.interactive:
        out_html = args.out_html or (
            root / "Results" / "hydrographs_future" / f"hydrograph_future_{basin_id}.html"
        )
        fig_html = build_overlay_hydrograph(
            sub,
            title,
            precip_label="Rainplusmelt (future)",
            q_sim_label="Simulated Q (future)",
            q_obs_label=None,
            q80_label=q80_legend,
            q50_label=q50_legend if "q50" in sub.columns else None,
            q20_label=q20_legend if "q20" in sub.columns else None,
        )
        write_interactive_html(fig_html, out_html)
        print(f"Saved {out_html} (open in a web browser to pan/zoom)")

    if not args.interactive or args.also_png:
        out_png.parent.mkdir(parents=True, exist_ok=True)
        _save_png(sub, title, q80_legend, out_png, args, q50_legend=q50_legend, q20_legend=q20_legend)


def _save_png(
    sub: pd.DataFrame,
    title: str,
    q80_legend: str,
    out_png: Path,
    args: argparse.Namespace,
    *,
    q50_legend: str = "",
    q20_legend: str = "",
) -> None:
    dates = sub["date"]
    precip = pd.to_numeric(sub["precip"], errors="coerce")
    q_sim = pd.to_numeric(sub["q_sim"], errors="coerce")
    q80 = pd.to_numeric(sub["q80"], errors="coerce")

    fig, ax_q = plt.subplots(1, 1, figsize=(args.fig_width, args.fig_height))
    ax_p = ax_q.twinx()
    ax_p.bar(
        dates,
        precip,
        color="#bcdffb",
        width=1.0,
        alpha=0.75,
        align="center",
        label="Rainplusmelt (future)",
        zorder=1,
    )
    ax_p.set_ylabel("P (mm d⁻¹)")
    ax_p.invert_yaxis()

    ax_q.plot(dates, q_sim, color="#0b2c85", linewidth=0.9, label="Simulated Q (future)", zorder=3)
    ax_q.plot(dates, q80, color="#c0392b", linewidth=1.0, linestyle="--", label=q80_legend, zorder=3)
    if "q50" in sub.columns:
        q50 = pd.to_numeric(sub["q50"], errors="coerce")
        ax_q.plot(dates, q50, color="#d68910", linewidth=1.0, linestyle=":", label=q50_legend, zorder=3)
    if "q20" in sub.columns:
        q20 = pd.to_numeric(sub["q20"], errors="coerce")
        ax_q.plot(dates, q20, color="#6c3483", linewidth=1.0, linestyle="-.", label=q20_legend, zorder=3)
    ax_q.set_ylabel("Q (mm d⁻¹)")
    ax_q.set_xlabel("Date")
    ax_q.set_title(title)
    ax_q.grid(alpha=0.25)
    ax_q.set_zorder(2)
    ax_q.patch.set_alpha(0.0)

    h1, l1 = ax_q.get_legend_handles_labels()
    h2, l2 = ax_p.get_legend_handles_labels()
    ax_q.legend(h1 + h2, l1 + l2, loc="upper right", framealpha=0.95)

    fig.tight_layout()
    fig.savefig(out_png, dpi=args.dpi)
    plt.close(fig)
    print(f"Saved {out_png}")


# Execute the script entry point
main()
